In [0]:
# if geopands is not installed run : 
#!pip install geopandas
# if issue occurs becuase of matplot lib run:
# %pip install "numpy<2" matplotlib
# then run 
# %pip install "numpy<2" matplotlib

In [0]:
import pandas as pd
import requests
import geopandas as gpd
from pathlib import Path

BASE_DIR = Path.cwd().parent
GEO_DIR = BASE_DIR / "data" / "raw" / "geo"

overlay_gdf = gpd.read_file(GEO_DIR / "tract_neighborhood_overlay.geojson")
neighborhoods_gdf = gpd.read_file(GEO_DIR / "stl_neighborhoods.geojson")
fema_gdf = gpd.read_file(GEO_DIR / "fema_flood_zones.geojson")

# 2024 ACS 5-year example
base_url = "https://api.census.gov/data/2024/acs/acs5"

In [0]:
# Missouri only; we will filter to St. Louis City (510) and St. Louis County (189)
params = {
    "get": ",".join([
        "NAME",
        "B19013_001E",  # median household income
        "B25070_001E",  # total renter households with gross rent as % of income
        "B25070_007E",  # 30.0 to 34.9 percent
        "B25070_008E",  # 35.0 to 39.9 percent
        "B25070_009E",  # 40.0 to 49.9 percent
        "B25070_010E",  # 50.0 percent or more
    ]),
    "for": "tract:*",
    "in": "state:29"
}

response = requests.get(base_url, params=params, timeout=60)
response.raise_for_status()

rows = response.json()
acs = pd.DataFrame(rows[1:], columns=rows[0])

# keep only St. Louis City + St. Louis County
acs = acs[acs["county"].isin(["510", "189"])].copy()

# make GEOID for joining to your tract overlay
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]

# numeric conversion
num_cols = [
    "B19013_001E",
    "B25070_001E",
    "B25070_007E",
    "B25070_008E",
    "B25070_009E",
    "B25070_010E",
]
for col in num_cols:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

# feature: % renter households spending >30% of income on gross rent
acs["housing_cost_burden_ratio"] = (
    acs["B25070_007E"]
    + acs["B25070_008E"]
    + acs["B25070_009E"]
    + acs["B25070_010E"]
) / acs["B25070_001E"]

# cleaner column names
acs = acs.rename(columns={
    "B19013_001E": "median_household_income",
    "B25070_001E": "rent_burden_total_households"
})

acs.head()

In [0]:
#Save CSV -- exists in repo 
# acs.to_csv("data/raw/census/acs_tract_stl.csv", index=False)

In [0]:
overlay_gdf = overlay_gdf.merge(
    acs[["GEOID", "median_household_income", "housing_cost_burden_ratio"]],
    on="GEOID",
    how="left"
)

overlay_gdf

#importance: each row is a piece of a census tract overlapping a neighborhood

In [0]:
overlay_gdf = overlay_gdf.dropna(subset=["NHD_NAME"])  # clean nulls

neighborhood_data = (
    overlay_gdf.groupby("NHD_NAME")
    .apply(lambda df: pd.Series({
        "income": (df["median_household_income"] * df["area_weight"]).sum() / df["area_weight"].sum(),
        "housing_cost_burden_ratio": (df["housing_cost_burden_ratio"] * df["area_weight"]).sum() / df["area_weight"].sum()
    }))
    .reset_index()
)

neighborhood_data.head()

In [0]:
fema_join = gpd.overlay(fema_gdf, neighborhoods_gdf, how="intersection")

fema_join["flood_area"] = fema_join.geometry.area
neighborhoods_gdf["total_area"] = neighborhoods_gdf.geometry.area


flood_pct = (
    fema_join.groupby("NHD_NAME_2")["flood_area"]
    .sum()
    .reset_index()
    .rename(columns={"NHD_NAME_2": "NHD_NAME"})
)

flood_pct = flood_pct.merge(
    neighborhoods_gdf[["NHD_NAME", "total_area"]],
    on="NHD_NAME",
    how="left"
)

flood_pct["flood_zone_pct"] = flood_pct["flood_area"] / flood_pct["total_area"]

flood_pct.head()

In [0]:
#combining everything 
final_df = neighborhood_data.merge(
    flood_pct[["NHD_NAME", "flood_zone_pct"]],
    on="NHD_NAME",
    how="left"
)

final_df.head()

neighborhoods_gdf = neighborhoods_gdf.merge(final_df, on="NHD_NAME")

neighborhoods_gdf.plot(column="housing_cost_burden_ratio", legend=True)

This map shows the housing cost burden ratio across St. Louis neighborhoods.
Higher values (yellow) indicate neighborhoods where a larger share of households
spend more than 30% of their income on housing. Lower values (purple) indicate
more affordable areas. The results were calculated using an area-weighted spatial
join between census tracts and neighborhood boundaries.

In [0]:
# Save final neighborhood-level dataset exists in gthtf
# final_df.to_csv("data/raw/census/neighborhood_data.csv", index=False)

In [0]:
print("Number of neighborhoods:", len(final_df))

All 79 neighborhoods have values after the spatial join.



In [0]:
print("census notebook completed successfully")

dbutils.notebook.exit("census completed")